In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from sqlalchemy import text
from src import config as C
from src.db_utils import run_sql_file

run_sql_file("05_model_comparison.sql")
engine = C.get_engine()

def wmape_by(col):
    q = f"""SELECT {col}, model,
                   ROUND(SUM(abs_err) / SUM(actual), 4) AS wmape
            FROM v_errors GROUP BY {col}, model"""
    return pd.read_sql(text(q), engine).pivot(index=col, columns="model", values="wmape")

for col in ["abc_class", "xyz_class", "store"]:
    print(f"\nWMAPE by {col}")
    print(wmape_by(col))


WMAPE by abc_class
model      lightgbm  moving_avg   naive  seasonal_naive
abc_class                                              
A            0.0413      0.0726  0.0614          0.0586
B            0.0464      0.0769  0.0674          0.0656
C            0.0564      0.0842  0.0801          0.0772

WMAPE by xyz_class
model      lightgbm  moving_avg   naive  seasonal_naive
xyz_class                                              
X            0.0396      0.0717  0.0595          0.0570
Y            0.0428      0.0740  0.0631          0.0605
Z            0.0565      0.0837  0.0796          0.0771

WMAPE by store
model  lightgbm  moving_avg   naive  seasonal_naive
store                                              
1        0.0460      0.0774  0.0660          0.0649
2        0.0418      0.0727  0.0601          0.0572
3        0.0405      0.0728  0.0616          0.0590
4        0.0436      0.0740  0.0656          0.0621
5        0.0502      0.0786  0.0730          0.0692
6        0.0509     

In [2]:
for col in ["abc_class", "xyz_class", "store"]:
    wmape_by(col).to_csv(C.OUT_DIR / f"05_wmape_by_{col}.csv")